In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from scipy.stats import ttest_ind, norm
from datetime import datetime
from scipy.stats import norm
import seaborn as sns
import random

import os

### Task 1.

**Задание**

В лекции мы выяснили, что в эксперименте «Refactoring backend» в экспериментальной группе среднее время загрузки увеличилось, а 99% квантиль уменьшился. Проверьте значимость отличий других квантилей.

Данные эксперимента «Refactoring backend»: 2022-04-13/2022-04-13T12_df_web_logs.csv и 2022-04-13/experiment_users.csv. Эксперимент проводился с 2022-04-05 по 2022-04-12. Измерения времени обработки запросов считаем независимыми. При проверке используйте нормальный доверительный интервал.

Выберете верные утверждения:
- Квантиль 0.70 статистически значимо изменился
- Квантиль 0.74 статистически значимо изменился
- Квантиль 0.78 статистически значимо изменился
- Квантиль 0.82 статистически значимо изменился
- Квантиль 0.86 статистически значимо изменился
- Квантиль 0.90 статистически значимо изменился
- Квантиль 0.95 статистически значимо изменился
- Квантиль 0.99 статистически значимо изменился
- Квантиль 0.999 статистически значимо изменился
- Квантиль 0.9999 статистически значимо изменился

**Решение**


In [8]:
# читаем лоокально
URL_BASE = ''

def read_database(file_name):
    return pd.read_csv(os.path.join(URL_BASE, file_name))

In [5]:
logs = read_database('2022-04-13T12_df_web_logs.csv')

In [11]:
logs['date'] = pd.to_datetime(logs['date'])

# ограничиваем временем эксперимента
logs = logs[
    (logs['date'] >= datetime(2022, 4, 5))
    & (logs['date'] < datetime(2022, 4, 12))
]
logs.head(3)

,user_id,page,date,load_time
2111340,190c2f,m,2022-04-05 00:00:38,66.2
2111341,e65269,m,2022-04-05 00:01:25,60.0
2111342,f0d8b6,b,2022-04-05 00:01:41,72.3


Читаем базу с пользователями. Колонка 'pilot' - вариант пользователя

In [70]:
df_users = read_database('experiment_users.csv')

df = pd.merge(df_users, logs, on='user_id', how='left')[['pilot', 'load_time']]

# сохранаяем время загрузки для двух вариантов
load_time_a = df[df['pilot'] == 0]['load_time'].values
load_time_b = df[df['pilot'] == 1]['load_time'].values

Задаем функицию для расчета нормального доверительного интервала

In [71]:
def get_ci_bootstrap_normal(boot_metrics: np.array, pe_metric: float, alpha: float=0.05):
    """Строит нормальный доверительный интервал.

    boot_metrics - значения метрики, полученные с помощью бутстрепа
    pe_metric - точечная оценка метрики
    alpha - уровень значимости
    
    return: (left, right) - границы доверительного интервала.
    """
    c = stats.norm.ppf(1 - alpha / 2)
    se = np.std(boot_metrics)
    left, right = pe_metric - c * se, pe_metric + c * se
    return left, right

Запустим процедуру бутстрепа для двух выборок. Для этого будем независимо семплировать из групп подвыборки такого же размера и считать бутстрепную оценку (разницу метрик). Повторим данную процедуру 1000 раз. Все это делаем в цикле по исследуемым значениям квантилей

In [72]:
q_list = [0.7, 0.74, 0.78, 0.82, 0.86, 0.90, 0.95, 0.99, 0.999, 0.9999]

In [74]:
for q in q_list:
    point_estimate = np.quantile(load_time_b, q) - np.quantile(load_time_a, q)
    bootstrap_values_a = np.random.choice(load_time_a, (1000, len(load_time_a)), True)
    bootstrap_metrics_a = np.quantile(bootstrap_values_a, q, axis=1)
    bootstrap_values_b = np.random.choice(load_time_b, (1000, len(load_time_b)), True)
    bootstrap_metrics_b = np.quantile(bootstrap_values_b, q, axis=1)
    bootstrap_stats = bootstrap_metrics_b - bootstrap_metrics_a
    # Строим нормальный доверительный инетрвал
    normal_ci = get_ci_bootstrap_normal(bootstrap_stats, point_estimate, 0.05)
    print(f'Significance for {q*100}% percentiles: {"significant" if normal_ci[0] * normal_ci[1] <= 0 else "not significant"}')

Significance for 70.0% percentiles: not significant
Significance for 74.0% percentiles: not significant
Significance for 78.0% percentiles: significant
Significance for 82.0% percentiles: not significant
Significance for 86.0% percentiles: not significant
Significance for 90.0% percentiles: not significant
Significance for 95.0% percentiles: not significant
Significance for 99.0% percentiles: not significant
Significance for 99.9% percentiles: significant
Significance for 99.99% percentiles: significant


### Task 2.

**Задание**

Реализуйте функцию run_bootstrap.

Шаблон решения

In [ ]:
import numpy as np
from scipy import stats


def run_bootstrap(bootstrap_metrics, pe_metric, alpha, bootstrap_ci_type):
    """Строит доверительный интервал и проверяет значимость отличий с помощью бутстрепа.
    
    :param bootstrap_metrics (np.array): множество значений статистики теста,
        посчитанные на бутстрепных выборках.
    :param pe_metric (float): значение статистики теста посчитанное по исходным данным.
    :param alpha (float): уровень значимости.
    :param bootstrap_ci_type (str): способ построения доверительного интервала.
        Возможные значения ['normal', 'percentile', 'pivotal'].
    :return ci, pvalue:
        ci [float, float] - границы доверительного интервала.
        pvalue (float) - 0 если есть статистически значимые отличия, иначе 1. Настоящее
        pvalue для произвольного способа построения доверительного интервала с помощью
        бутстрепа вычислить не тривиально. Будем использовать краевые значения 0 и 1.
    """
        # YOUR_CODE_HERE

Обратите внимание, что на вход функции подаются не исходные значения метрики, а множество значений статистики теста, посчитанные на бутстрепных выборках. Это позволит нам детерминировано протестировать правильноcть решения. Самостоятельно бутстрепить данные внутри функции run_bootstrap не нужно.

Пример реализации функции для вычисления bootstrap_metrics

In [ ]:
def generate_bootstrap_metrics(data_one, data_two, bootstrap_iter, bootstrap_agg_func):
    """Генерирует значения метрики, полученные с помощью бутстрепа.

    :param data_one, data_two (np.array): значения метрик в группах.
    :param design (Design): объект с данными, описывающий параметры эксперимента
    :param bootstrap_iter (int): количество итераций бутстрепа.
    :param bootstrap_agg_func (str): метрика эксперимента.
        Возможные значения ['mean', 'quantile 95'].
    :return bootstrap_metrics, pe_metric:
        bootstrap_metrics (np.array) - множество значений статистики теста,
            посчитанные на бутстрепных выборках.
        pe_metric (float) - значение статистики теста посчитанное по исходным данным.
    """
    bootstrap_data_one = np.random.choice(data_one, (len(data_one), bootstrap_iter))
    bootstrap_data_two = np.random.choice(data_two, (len(data_two), bootstrap_iter))
    if bootstrap_agg_func == 'mean':
        bootstrap_metrics = (
            bootstrap_data_two.mean(axis=0) - bootstrap_data_one.mean(axis=0)
        )
        pe_metric = data_two.mean() - data_one.mean()
        return bootstrap_metrics, pe_metric
    elif bootstrap_agg_func == 'quantile 95':
        q = 0.95
        bootstrap_metrics = (
            np.quantile(bootstrap_data_two, q, axis=0)
            - np.quantile(bootstrap_data_one, q, axis=0)
        )
        pe_metric = np.quantile(data_two, q) - np.quantile(data_one, q)
        return bootstrap_metrics, pe_metric
    else:
        raise ValueError('Неверное значение bootstrap_agg_func')
        
data_one, data_two = np.array([1, 3]), np.array([5, 7])
bootstrap_iter = 10
bootstrap_agg_func = 'mean'
bootstrap_metrics, pe_metric = generate_bootstrap_metrics(
    data_one, data_two, bootstrap_iter, bootstrap_agg_func
)
# bootstrap_metrics = np.array([6., 5., 3., 4., 5., 2., 6., 4., 4., 4.])
# pe_metric = 4.0

Пример применения функции run_bootstrap

In [ ]:
bootstrap_metrics = np.arange(-90, 910)
pe_metric = 600.
alpha = 0.05
bootstrap_ci_types = ['normal', 'percentile', 'pivotal']
for bootstrap_ci_type in bootstrap_ci_types:
    ci, pvalue = run_bootstrap(bootstrap_metrics, pe_metric, alpha, bootstrap_ci_type)
    print(bootstrap_ci_type)
    print(f'ci = {np.array(ci).round()}, pvalue = {pvalue}')
# >>> normal: ci = [  34. 1166.], pvalue = 0.0
# >>> percentile: ci = [-65. 884.], pvalue = 1.0
# >>> pivotal: ci = [ 316. 1265.], pvalue = 0.0

**Решение**

In [79]:
import numpy as np
from scipy import stats


def run_bootstrap(bootstrap_metrics, pe_metric, alpha, bootstrap_ci_type):
    """Строит доверительный интервал и проверяет значимость отличий с помощью бутстрепа.
    
    :param bootstrap_metrics (np.array): множество значений статистики теста,
        посчитанные на бутстрепных выборках.
    :param pe_metric (float): значение статистики теста посчитанное по исходным данным.
    :param alpha (float): уровень значимости.
    :param bootstrap_ci_type (str): способ построения доверительного интервала.
        Возможные значения ['normal', 'percentile', 'pivotal'].
    :return ci, pvalue:
        ci [float, float] - границы доверительного интервала.
        pvalue (float) - 0 если есть статистически значимые отличия, иначе 1. Настоящее
        pvalue для произвольного способа построения доверительного интервала с помощью
        бутстрепа вычислить не тривиально. Будем использовать краевые значения 0 и 1.
    """
    
    if bootstrap_ci_type == 'normal':
        z = stats.norm.ppf(1-alpha/2)
        se = np.std(bootstrap_metrics)
        ci_left, ci_right = pe_metric - z * se, pe_metric + z * se
    
    elif bootstrap_ci_type == 'percentile':
        ci_left, ci_right = np.quantile(bootstrap_metrics, [alpha / 2, 1 - alpha / 2])

    elif bootstrap_ci_type == 'pivotal':
        ci_left, ci_right = 2 * pe_metric - np.quantile(bootstrap_metrics, [1 - alpha / 2, alpha / 2])
    else:
        raise ValueError("bootstrap_ci_type is to be in ['normal', 'percentile', 'pivotal']")
        
    if not (ci_left < 0 < ci_right):
        pvalue = float(0)
    else:
        pvalue = float(1)
    return [ci_left, ci_right], pvalue

In [80]:
bootstrap_metrics = np.arange(-90, 910)
pe_metric = 600.
alpha = 0.05
bootstrap_ci_types = ['normal', 'percentile', 'pivotal']
for bootstrap_ci_type in bootstrap_ci_types:
    ci, pvalue = run_bootstrap(bootstrap_metrics, pe_metric, alpha, bootstrap_ci_type)
    print(bootstrap_ci_type)
    print(f'ci = {np.array(ci).round()}, pvalue = {pvalue}')
# >>> normal: ci = [  34. 1166.], pvalue = 0.0
# >>> percentile: ci = [-65. 884.], pvalue = 1.0
# >>> pivotal: ci = [ 316. 1265.], pvalue = 0.0

normal
ci = [  34. 1166.], pvalue = 0.0
percentile
ci = [-65. 884.], pvalue = 1.0
pivotal
ci = [ 316. 1265.], pvalue = 0.0
